# YOLOv8 + ArcFace + SQLite

This notebook demonstrates:
- YOLOv8 for person detection
- ArcFace embeddings via DeepFace for face recognition
- SQLite to store identities and log recognition events


In [ ]:
# Minimal sanity check
import sqlite3, cv2, numpy as np
from ultralytics import YOLO
import tensorflow as tf
import deepface
print('OpenCV:', cv2.__version__)
print('TensorFlow:', tf.__version__)
print('DeepFace:', deepface.__version__)
print('Notebook pipeline ready')

## Config

In [ ]:
ENROLL_DIR = 'enroll'
DB_PATH = 'surveillance.db'
YOLO_MODEL = 'yolov8n.pt'
DET_CONF = 0.35
SIM_THRESH = 0.35
print('Config ready:', ENROLL_DIR, DB_PATH, YOLO_MODEL)

## SQLite Schema

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute('''CREATE TABLE IF NOT EXISTS identities(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT UNIQUE,
  embedding BLOB)''')
cur.execute('''CREATE TABLE IF NOT EXISTS events(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  ts TEXT,
  cam TEXT,
  name TEXT,
  distance REAL,
  note TEXT)''')
cur.execute('CREATE INDEX IF NOT EXISTS idx_events_ts ON events(ts)')
cur.execute('CREATE INDEX IF NOT EXISTS idx_events_name ON events(name)')
conn.commit()
print('[INFO] SQLite ready')

## Helpers

In [ ]:
import pickle
from datetime import datetime, timezone
from deepface import DeepFace

def get_arcface_embedding(img_rgb, enforce=True):
    try:
        rep = DeepFace.represent(img_path=img_rgb, model_name='ArcFace',
                                detector_backend='retinaface', enforce_detection=enforce)
        if isinstance(rep, list) and rep and 'embedding' in rep[0]:
            return np.array(rep[0]['embedding'], dtype=np.float32)
    except Exception:
        pass
    return None

def cosine_distance(a, b):
    den = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return 1.0 - float(np.dot(a, b) / den)

def enroll_identity(name, image_paths):
    embs = []
    for p in image_paths:
        img_bgr = cv2.imread(p)
        if img_bgr is None:
            continue
        emb = get_arcface_embedding(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), enforce=True)
        if emb is not None:
            embs.append(emb)
    if not embs:
        print('[ERROR] no embeddings for', name); return False
    mean_emb = np.mean(np.stack(embs), axis=0)
    blob = pickle.dumps(mean_emb, protocol=pickle.HIGHEST_PROTOCOL)
    cur.execute('INSERT OR REPLACE INTO identities(name, embedding) VALUES(?,?)', (name, blob))
    conn.commit(); print('[OK] enrolled', name); return True

def load_identities_from_db():
    cur.execute('SELECT name, embedding FROM identities')
    gallery = {}
    for name, blob in cur.fetchall():
        try: gallery[name] = pickle.loads(blob).astype(np.float32)
        except: pass
    return gallery

def recognize_face(face_rgb, gallery, sim_thresh=SIM_THRESH):
    probe = get_arcface_embedding(face_rgb, enforce=False)
    if probe is None or not gallery:
        return 'Unknown', None
    best_dist, best_name = 1e9, None
    for name, emb in gallery.items():
        d = cosine_distance(probe, emb)
        if d < best_dist:
            best_dist, best_name = d, name
    return (best_name, best_dist) if best_dist <= sim_thresh else ('Unknown', best_dist)

def log_event(cam_name, name, dist, note):
    ts = datetime.now(timezone.utc).isoformat()
    cur.execute('INSERT INTO events(ts, cam, name, distance, note) VALUES(?,?,?,?,?)',
                (ts, cam_name, name, float(dist) if dist else None, note))
    conn.commit()

## Live loop

In [ ]:
def detect_and_recognize(source=0, cam_name='webcam', use_db=True):
    model = YOLO(YOLO_MODEL)
    gallery = load_identities_from_db() if use_db else {}
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print('[ERROR] cannot open', source); return
    while True:
        ok, frame = cap.read()
        if not ok: break
        res = model.predict(frame, conf=DET_CONF, verbose=False)[0]
        if res.boxes is not None:
            for b in res.boxes:
                if int(b.cls[0].item()) != 0: continue
                x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                roi = frame[y1:y2, x1:x2]
                if roi.size == 0: continue
                name, dist = recognize_face(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB), gallery)
                label = f"{name} ({dist:.2f})" if dist else name
                cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                cv2.putText(frame, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,(0,255,0),2)
                log_event(cam_name, name, dist, 'recognized' if name!='Unknown' else 'unknown')
        cv2.imshow('YOLOv8+ArcFace', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release(); cv2.destroyAllWindows()